# Heralded single-photon imaging: troubleshooting runs (PI-MAX4 emICCD)

**Goal.** Take camera runs with full metadata, and make simple, labelled plots we can put in front of the group to work out why the heralded signal is hard to see.

**Setup.** ppKTP Sagnac SPDC source pumped at 405 nm. The idler APD output goes straight to the camera's TRIGGER IN. The signal arm has roughly 110–130 m of fibre as a delay line, then goes to the camera.

**What a "run" is.** One call to `take_run(...)` configures the camera, reads every setting back, takes a number of readouts, and saves everything to one `.npz` file:
- the summed full frame,
- the first raw readout,
- the ROI crop of every readout,
- the settings asked for **and** what the camera reported back.

If the camera refuses a setting, the run stops before taking data, so nothing is saved under a misleading label.

**Everything here runs remotely.** No step needs anyone in the lab. Instead of blocking the signal arm for a dark frame, the backgrounds come from the camera itself:
- **Off-delay background.** The same run with the gate moved 5 µs later, away from the heralded photon. It still sees the unheralded light, dark counts, pedestal and hot pixels. So signal minus this background is the **heralded** part only, which is exactly what we want to see.
- **Window comparisons.** The coarse scan, the halving ladder and the delay scans compare time windows against each other, so they need no background at all.
- **The ring around the spot.** Every spot number subtracts the median of a ring around the beam in the same readout.

**Fastest route: section 2b** runs the whole search unattended and writes a report with figures. The sections after it do the same steps one at a time, so you can look in between.

Runs go to `Desktop\HOI_freshstaret\data\raw\troubleshoot\`, and figures go to `...\data\processed\troubleshoot_figures\`. `compare_runs.ipynb` rebuilds every plot from those files **without the camera**.

### Safety, every session
- **Close LightField first.** Only one program can hold the camera.
- The photocathode is damaged by *integrated* light with the intensifier on. Keep the room dark, close the enclosure, and confirm the pump is blocked.
- Watch **saturated px** after every run. If it is above zero, lower the gain or the accumulations.

## Findings so far (20 Oct)

These come from the first session's saved runs. The plots behind each one can be rebuilt in `compare_runs.ipynb`.

1. **The beam reaches the camera.** At a 100 µs gate there is a compact spot at sensor x ≈ 501, y ≈ 454, about 15 px across. Per readout it gives about 100 000 net ADU, 51σ above the blocked-arm dark. It also gives about 180 photon hits, against 6 in the dark. The camera's own photon thresholding gives the same hit count as the software, so the threshold logic is consistent.
2. **Everything else on the sensor is not from the signal arm.** The second bright spot at x ≈ 750, the glow on the left, the vertical streaks and a faint ring are all in the dark too.
3. **No heralded excess yet.** The early and late halves of 300–2300 ns are identical in three repeats, and a 10 ns scan over 1270–1315 ns is flat. Photon hits in the spot at narrow gates are about 0.2–0.5 per readout, about the dark level.
4. **The trigger rate looks far higher than 15 kHz.** At a 100 µs gate, 5000 gates took 0.512 s per readout. 0.500 s of that is gate-open time, so the camera re-triggered within ~2 µs of every gate closing. At narrow gates the readout time limits the speed, giving at least 22 k triggers per second. Section 10 measures it properly.
5. **There is a glow in the spot that doesn't depend on gate width.** At 10 ns and 1 µs the spot still shows ~1500–2500 ADU per readout, but with almost no photon hits. Section 13 measures this with matching darks.

**Code changes since the first session:**
- **Settings are checked.** Every setting is read back, and a run stops if the camera did not take one. `check_settable` shows the real error.
- **Spot metrics.** Numbers are taken in a 10 px circle around the beam, minus a surrounding ring. Whole-ROI numbers diluted the spot about 100 times, which is why the first figure said "2.9 sigma".
- **Thresholds.** Photon thresholds are now ADU **above each readout's own pedestal**. One value works at every gate width, even though the pedestal moved from ~694 to ~640 ADU.
- **Trigger rate.** Each run prints what its timing says about the trigger rate.

## What the references did, and what we are aiming for

From the two theses and the PRA in `Desktop\HOI_freshstaret`. The Schwarz *et al.* paper (PRA 101, 043815) is Cameron's chapter 3 in paper form.

| Setting | Target | Where it comes from |
|---|---|---|
| Sensor temperature | −20 °C, wait for **Locked** | Cameron p. 82, Xu p. 32 |
| Idler trigger rate | ~15 kHz (camera trigger tops out at 20 kHz) | Cameron p. 80 |
| Trigger threshold | 0.5 V (default 1.5 V never fires) | Xu p. 33 |
| Gate width | 3 ns (= the coincidence window) | Cameron p. 81, Xu p. 33 |
| Finding the delay | start at a 100 ns gate, find the beam, halve the gate, repeat | Cameron p. 81 |
| emICCD gain | 9000 | Cameron p. 81–82, Xu p. 34 |
| On-CCD accumulations | 50 000 per readout | Xu p. 34 |
| Readouts summed ("exposures per frame") | 2000 for final images, 200 for quick checks | Xu p. 34, Cameron p. 81 |
| ROI | ~150 × 150 px (Cameron used 140 × 140) | Xu p. 34, Cameron p. 92 |
| Background | same settings, signal arm blocked, pixel-by-pixel subtraction, retake when the idler rate drifts. **Here:** an off-delay background, so no one needs to be in the lab | Cameron p. 82 |
| Post-processing | two-pass Wiener filter with a 9 × 9 window | Xu p. 35 |
| SNR | mean(signal − background) / std(background) | Cameron Table 3.1 |

**Their figures.** The theses show accumulated, background-subtracted images of about 1 h each (Cameron Fig. 3.6, Fig. 4.2), before-and-after post-processing (Xu Fig. 4.2), and an SNR table (Cameron Table 3.1). They do **not** show pixel histograms, single raw readouts or profiles. Those are troubleshooting additions.

**Fibre.** Cameron used 30 m, about 144 ns. You have 110–130 m, so the first estimate is about 450–580 ns (section 3). That estimate is only as good as its assumptions, which is why section 14 also scans the whole 0–10 µs range.

In [ ]:
import sys, importlib
sys.path.insert(0, r"C:\Users\ReschLab\Single-Photon-Imaging")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import hoi_troubleshoot as ht          # acquisition + plotting helpers (this repo)
importlib.reload(ht)                   # pick up edits to the module without restarting
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 250)

SPOT = (501, 454)      # beam position on the sensor (x, y) from 20 Oct; updated in section 9
HIT_THR = 200          # photon threshold, ADU above pedestal, from 20 Oct; updated in section 12
print("runs will be saved to:", ht.DATA_DIR)
print("figures will be saved to:", ht.FIG_DIR)

## 1. Open the camera

This uses `camera_control.py` from `HOI_freshstaret\src`, the hardware-tested code. It sets the gate through the raw PICam call because pylablib silently ignores it, and it patches PICam timing bugs. It prints the allowed gate range: delay ≥ 27 ns, width ≥ 2.68 ns.

"already opened" or a `RecursionError` means another kernel or LightField still holds the camera.

In [ ]:
try:
    cam
    print("camera already open in this kernel:", cam.get_device_info())
except NameError:
    cam, GATE_LIMITS = ht.open_camera()

## 2. Cool to −20 °C, show the state, and check that settings can be written

Do not take data until the cooling cell says **LOCKED**.

The last call re-sends the *current* value of every trigger, gating and photon-detection setting, so it changes nothing. It prints the camera's real error for any setting it refuses. On 20 Oct those settings printed FAIL, but the read-back showed the values were already right, except for the internal-trigger control, which never switched. If anything prints FAIL here, copy the error into the group notes.

In [ ]:
ht.cool_down(cam, setpoint_c=-20.0)
ht.show_state(cam)
print("\nCan each setting be written?")
ht.check_settable(cam)

## 2b. Unattended hunt: find the heralded photons and image them

`photon_hunt.py` runs the whole search with no one in the lab, logging every decision to a `report.md` with numbered figures:

1. Trigger-rate test, from the camera's timing.
2. Wide-gate reference. It stops if the beam spot isn't on the sensor.
3. **Coarse delay scan** with 1 µs gates from 27 ns to 10 µs. It accepts a window only if it stands out from all the others by at least 5σ. If nothing does, it takes a second pass with three times the readouts before giving up.
4. **Halving ladder** from that window down to ~10 ns. If a step is unclear, it scans across the window to catch photons straddling the boundary.
5. **Fine delay scan** with a 3 ns gate to find the peak.
6. **Target image**: gain 9000, 3 ns gate, 50 000 accumulations, 200 readouts, plus an off-delay background. It then makes the six-panel figure, the Wiener view and a threshold sweep.

It takes roughly 20–40 min, depending on the trigger rate. It stops early, with recommendations, if no window stands out. Interrupt the kernel to stop it. The hunt uses the camera already open in this notebook and leaves it open afterwards.

The same hunt runs from a terminal with `python photon_hunt.py`, or through the `photon-hunter` Claude agent. `python photon_hunt.py --help` lists every option. The terminal and agent versions close the camera at the end.

In [ ]:
import photon_hunt as ph
importlib.reload(ph)
HUNT_ARGS = ph.parse_args([
    "--idler-khz", "15",            # <- type the idler counter reading (goes in the report)
    "--target-readouts", "200",     # 0 = stop after finding the delay
])
hunt_result, hunt_folder = ph.hunt(cam=cam, args=HUNT_ARGS)
print(hunt_result)
print("report:", hunt_folder / "report.md")

## 3. Where in time should the heralded photon be?

> gate delay ≈ (signal fibre − idler fibre) × n / c − APD latency − trigger cable − camera insertion delay

- Fibre: about 4.9 ns per metre.
- APD output latency: about 15–30 ns. Check the APD datasheet.
- BNC cable: about 5 ns per metre.
- PI-MAX4 insertion delay, from Trigger In to the photocathode gate: 35 ns (manual §6).

Edit the numbers to match the bench.

In [ ]:
lo, hi = ht.expected_delay_ns(
    signal_fiber_m=(110, 130),   # extra fibre on the signal arm (range = how sure we are)
    idler_fiber_m=0,             # idler fibre to the APD, if you want to subtract it
    apd_latency_ns=(15, 30),
    trigger_cable_m=(1, 5),      # APD -> camera TRIGGER IN cable length
    camera_insertion_ns=35,
)
print(f"Expected gate delay: roughly {lo:.0f} to {hi:.0f} ns after the trigger")

## 4. Why a huge gate "works", and what to expect at 3 ns

A 100 µs gate collects **every** signal-arm photon in those 100 µs, heralded or not. So a signal there proves light reaches the camera, not that the timing or the heralding works. As the gate shrinks around the right delay, the unheralded light falls in proportion to the width while the heralded photons stay. Type in the counter readings. The estimate assumes the delay is right **and** that every camera trigger is a real idler photon. Finding 4 suggests the second assumption may be wrong.

In [ ]:
print("Expected photoelectrons per readout (assumes the delay is right):")
display(ht.expected_counts(
    idler_khz=15.0,            # idler singles on the counter
    signal_khz=18.0,           # signal singles on the counter (same fibre that goes to the camera)
    coinc_khz=1.5,             # coincidences
    signal_apd_eff=0.6,
    transmission_to_camera=1.0, # optics between fibre output and camera
    qe=0.25,                   # photocathode QE (Cameron p. 82)
    accumulations=5000,
).round(3))

## 5. Settings for this session

Every run starts from `BASE` and overrides only what it changes.

| Key | What it is |
|---|---|
| `gain` | emICCD gain (1–10 000). Target 9000. |
| `gate_delay_ns`, `gate_width_ns` | When the intensifier opens after the trigger, and for how long. |
| `accumulations` | Gated exposures summed **on the CCD** before one readout, one per trigger. LightField calls this "On-CCD Accumulations". |
| `n_readouts` | Readouts summed **in software**. LightField calls this "Exposures per Frame". |
| `trigger_*` | External trigger from the idler APD: 0.5 V, rising edge, 50 Ω, DC. |
| `photon_mode`, `photon_threshold` | Camera photon detection. Keep `Disabled` (raw ADU), so every threshold can be tried in software afterwards. |
| `roi_center`, `roi_size` | ROI centre (x, y) on the sensor, and its size. |
| `idler_rate_khz` | **Type the counter reading.** It is saved with each run. |
| `notes` | Free text saved with the run. |

In [ ]:
BASE = dict(
    gain=9000,
    gate_delay_ns=309,
    gate_width_ns=100_000,     # 100 us, as in LightField
    accumulations=5000,
    n_readouts=20,
    trigger_source="External",
    trigger_threshold_v=0.5,
    trigger_edge="Rising Edge",
    trigger_termination="Fifty Ohms",
    trigger_coupling="DC",
    photon_mode="Disabled",
    roi_center=(500, 455),
    roi_size=150,
    idler_rate_khz=None,       # <- type the counter reading
)

## 6. Run A: the wide gate where LightField showed something

About 12 s. Interrupt the kernel at any time and the readouts taken so far are still saved. Each run prints what its timing says about the trigger rate.

In [ ]:
runA = ht.take_run(cam, label="A_wide_gate", kind="signal", **BASE,
                   notes="LightField settings reproduced")

## 7. The per-run figures

A 100 µs gate is too wide for an off-delay background, so Run A is shown **minus its own median**. The spot numbers still subtract the ring around the beam. The 20 Oct blocked-arm dark showed that everything outside the central spot is there with the signal arm blocked too: the second spot at x ≈ 750, the glow and the streaks. Only the central spot is signal-arm light.

**Six-panel figure.** The title states all settings and the **spot** numbers, from a 10 px circle around the beam minus the ring around it.

- **(a) Full frame.** Mean ADU per readout. The cyan box is the ROI.
- **(b) ROI zoom.** Raw, no subtraction.
- **(c) Net image.** Signal minus background, or minus the median when there is no background run. White is zero and red is above it. The solid circle is the spot aperture. The dotted ring is the local background.
- **(d) Pixel values above pedestal.** Each pixel in each *single* readout, minus that readout's median. The narrow peak at 0 is readout noise. Photon hits are the tail on the right. When a background run is given, it is drawn in black, and **the red excess over black is the signal**.
- **(e, f) Row and column profiles.** Through the spot.

In [ ]:
ht.plot_run(runA, None, threshold=HIT_THR, spot=SPOT, save_as="A_wide_gate")
ht.plot_raw_vs_accumulated(runA, None, save_as="A_wide_gate_raw_vs_sum")

## 8. Locate the beam: spot position and ROI centre

`find_spot` gives the beam position used by every spot number below. `suggest_roi_center` does the same on the full frame, for centring the ROI. Change `BASE["roi_center"]` only if the beam is near the ROI edge. The saved ROI crops of earlier runs cannot be moved.

In [ ]:
SPOT = ht.find_spot(runA, None)
print("spot (x, y) =", SPOT)
cx, cy = ht.suggest_roi_center(runA, None)
# BASE["roi_center"] = (cx, cy)        # uncomment only if the beam sits near the ROI edge

## 9. A low-light reference run (replaces the blocked-arm dark)

This is the same as Run A but with a **10 ns** gate. That lets in about 10 000 times less unheralded light, while the pedestal, readout noise and hot pixels stay the same. It stands in for a dark frame when choosing the photon threshold in section 12. It isn't a perfect dark: dark counts and any heralded photons at 309 ns could still show up. Its timing also gives a second trigger-rate estimate.

In [ ]:
refLow = ht.take_run(cam, label="A_low_light_10ns", kind="dark_lowlight", **{**BASE, "gate_width_ns": 10})

## 10. How fast is the camera actually being triggered?

The camera can't count triggers, but it can time them. With a tiny gate and many accumulations, each readout has to wait for that many triggers, so the trigger rate ≈ accumulations ÷ time per readout.

**Read the idler counter at the same time.**
- **About the same** (≈ 15 kHz): triggering is fine.
- **Much higher**: the camera is firing on something besides idler photons, such as noise or ringing on the cable, double pulses, or a threshold too close to the noise. Each spurious trigger opens a gate with no partner photon, so heralded photons get diluted or lost. Check the TRIGGER IN signal on a scope with 50 Ω termination, and try raising the threshold, for example to 0.8–1.0 V if the APD pulse is taller.

On 20 Oct the timing implied at least 22 kHz, and more likely hundreds of kHz.

In [ ]:
TRIG_RATE = ht.measure_trigger_rate(cam, BASE, accumulations=100_000, width_ns=3, n_readouts=4)

## 11. Control: internal trigger

This repeats Run A with the camera's **internal** trigger at 5 kHz instead of the idler. If the spot is just as bright per gate, the wide-gate signal is unheralded light. `take_run` now refuses to run if the camera stays on the external trigger, which is what happened on 20 Oct. If that happens, the error lists what didn't apply. Run `ht.check_settable(cam)` and note the error.

In [ ]:
runA_int = ht.take_run(cam, label="A_internal_trigger", kind="internal",
                       **{**BASE, "trigger_source": "Internal", "internal_trigger_hz": 5000},
                       notes="control: internal trigger, same gate")
ht.plot_compare([(runA, None), (runA_int, None)], ncols=2, spot=SPOT, save_as="A_external_vs_internal")
cmp = ht.summary_table([(runA, None), (runA_int, None)], spot=SPOT, threshold=HIT_THR)
cmp["spot ADU per 1k gates"] = cmp["spot ADU/readout"] / (cmp["accum"] / 1000)
display(cmp)

## 12. Photon threshold

A photon hit at high gain is a splash a few hundred to a few thousand ADU above the pedestal. Readout noise is about 30–70 ADU. Counting pixels above a threshold throws away readout noise and EM excess noise.

This all runs in software on readouts already saved, so it needs no new data. It compares Run A with the 10 ns low-light reference from section 9. Thresholds are **ADU above each readout's own pedestal**, and hits are counted in the spot aperture. The best value becomes `HIT_THR` for the rest of the notebook. On 20 Oct it was 200 ADU. The camera's own Photon Detection Thresholding gave the same hit count as the software at the equivalent absolute value, so there is no need to repeat it on the camera.

In [ ]:
thr_table, HIT_THR = ht.threshold_sweep(runA, refLow, spot=SPOT, save_as="A_threshold_sweep")
print(f"photon threshold: {HIT_THR:g} ADU above pedestal")
display(thr_table.round(2).iloc[::5])

## 13. Gate-width series: does the spot scale with gate time?

This keeps the delay fixed and steps the gate width from 100 µs down to 10 ns. The plot shows spot ADU and spot photon hits against **gate-open time per readout** (accumulations × width), on log-log axes:

- **Unheralded light and dark counts** arrive at random times, so they follow the dashed proportional line.
- **A flat floor at small widths** is signal that doesn't depend on the gate, such as light leaking past the closed intensifier.
- **Heralded photons** would add a constant once the gate covers the right delay. At a fixed delay of 309 ns, the narrow gates may miss them, which is expected.

On 20 Oct a rough version of this plot put 1 µs hits about 10 times below the proportional line, and the spot ADU did not fall below ~1500. This measures it properly at a single delay. It takes about 5 widths × 20 readouts, roughly 1–2 min.

In [ ]:
WIDTHS_NS = [100_000, 10_000, 1000, 100, 10]
ws_sig = ht.width_series(cam, BASE, WIDTHS_NS, delay_ns=309, label="wseries")
ht.plot_width_series(ws_sig, threshold=HIT_THR, spot=SPOT, save_as="width_series")
display(ht.summary_table([(r, None) for r in ws_sig], spot=SPOT, threshold=HIT_THR))

## 14. Coarse delay scan: where in 0–10 µs is the herald?

This uses 1 µs gates stepped across 27 ns to 10 µs, about 1 min in total. Unheralded light is the same in every window, so **only the window that contains the heralded photons stands out** in the spot signal and spot hits. It covers the whole range, so it doesn't rely on the section 3 estimate. If one window stands out, start the halving ladder (section 15) from that window.

**Independent check without the camera.** This is what Sacha Shwartz suggested. Put the far end of the 120 m fibre on an APD and histogram idler–signal coincidences against delay on the time tagger. That gives the true relative delay to about 1 ns, and the real heralding efficiency. The camera gate delay is then roughly that delay minus the camera's 35 ns insertion delay, minus the trigger-cable delay. If no coincidence peak shows up there either, the problem is in the source or the fibres, not the camera.

In [ ]:
COARSE_DELAYS = np.arange(27, 10_000, 1000)
coarse = ht.delay_scan(cam, BASE, COARSE_DELAYS, 1000, n_readouts=20, label="coarse")
ht.plot_delay_scan(coarse, threshold=HIT_THR, spot=SPOT, save_as="coarse_delay_scan")

## 15. Shrink the gate: the halving ladder

Each step takes **two runs that split the current window into an early half and a late half**. Both halves have the same width, so the same unheralded light, dark counts and pedestal. **Any difference is heralded photons.** The step scores the spot aperture with its ring subtracted, which cancels pedestal drift, plus photon hits in the spot. It keeps the brighter half when |z| ≥ 3.

- **Starting window.** Use the window that stood out in section 14. `(300, 2000)` was tried three times on 20 Oct and came back unclear every time.
- **"unclear"** means there is no heralded excess in either half, or it straddles the boundary. Don't keep repeating the same window. Go back to section 14, or to the time-tagger check.

**Manual mode.** Run the cell, look, and run it again. It moves to the chosen half automatically.

In [ ]:
LADDER = dict(delay=300.0, width=2000.0)      # <- set from the coarse scan

In [ ]:
LADDER_READOUTS = 20
a, b = ht.take_halves(cam, BASE, LADDER["delay"], LADDER["width"], n_readouts=LADDER_READOUTS)
res = ht.compare_halves(a, b, threshold=HIT_THR, spot=SPOT,
                        save_as=f"ladder_{LADDER['delay']:g}_{LADDER['width']:g}")
if res["pick"] != "unclear":
    LADDER = dict(delay=res["next_delay_ns"], width=res["next_width_ns"])
print("next window:", LADDER)

**Automatic mode.** This repeats the step until the camera's minimum width, or the first unclear step. It is off by default. Turn it on only after a manual step has picked a half with |z| well above 3.

In [ ]:
RUN_AUTO_LADDER = False
if RUN_AUTO_LADDER:
    while LADDER["width"] / 2 >= GATE_LIMITS["width_min_ns"]:
        a, b = ht.take_halves(cam, BASE, LADDER["delay"], LADDER["width"], n_readouts=LADDER_READOUTS)
        res = ht.compare_halves(a, b, threshold=HIT_THR, spot=SPOT,
                                save_as=f"ladder_{LADDER['delay']:g}_{LADDER['width']:g}")
        print(LADDER, "->", res["pick"], f"z={res['z']:.1f}")
        if res["pick"] == "unclear":
            break
        LADDER = dict(delay=res["next_delay_ns"], width=res["next_width_ns"])
    print("final window:", LADDER)

## 16. Fine delay scan at a fixed gate width

Once a window of about 30 ns or less is known, scan across it in steps of about half the width. A peak in spot hits against delay is the cleanest single slide proving the heralding works. Its width tells you the timing jitter.

In [ ]:
SCAN_WIDTH_NS = 10
centre = LADDER["delay"] + LADDER["width"] / 2
SCAN_DELAYS = np.arange(centre - 30, centre + 30 + 1, 5)
scan_runs = ht.delay_scan(cam, BASE, SCAN_DELAYS, SCAN_WIDTH_NS, n_readouts=20)
ht.plot_delay_scan(scan_runs, threshold=HIT_THR, spot=SPOT, save_as=f"delay_scan_{SCAN_WIDTH_NS}ns")

## 17. Target settings: gain 9000, 3 ns gate, 150 px ROI, 50 000 accumulations

Set `TARGET_DELAY_NS` from the peak of section 16.

| Readouts | Wall time at 15 kHz | Use |
|---|---|---|
| 20 | ~1.2 min | first look |
| 200 | ~12 min | Xu's quick check |
| 2000 | ~2 h | thesis-quality image |

The background is the **off-delay** version of the same run: the gate moves 5 µs later and everything else stays the same. Signal minus background is then only the heralded photons. It uses half as many readouts and is rescaled automatically.

Then three figures: the six-panel figure, the Xu two-pass Wiener view with the Cameron SNR before and after, and a threshold sweep.

In [ ]:
TARGET_DELAY_NS = LADDER["delay"]      # <- put the delay-scan peak here
TARGET = {**BASE, "gate_delay_ns": TARGET_DELAY_NS, "gate_width_ns": 3, "accumulations": 50_000,
          "n_readouts": 20}
runT = ht.take_run(cam, label="T_target", kind="signal", **TARGET)
darkT = ht.take_offdelay_background(cam, runT, shift_ns=5000, n_readouts=max(TARGET["n_readouts"] // 2, 10))
ht.plot_run(runT, darkT, threshold=HIT_THR, spot=SPOT, save_as="T_target")
ht.plot_wiener(runT, darkT, window=9, save_as="T_target_wiener")
thr_table_T, thr_T = ht.threshold_sweep(runT, darkT, spot=SPOT, save_as="T_target_threshold_sweep")

## 18. Compare every run from this session

Each signal run is paired with the most recent dark that has the same gain, gate width, accumulations, photon settings and ROI.

- **s / readout and gates/s (>=)** come from the timing. Gates per second is a lower bound on the trigger rate.
- **spot net ADU/readout and spot sigma** are the beam signal in the 10 px circle, minus its ring, against the dark.
- **spot hits/readout and hits sigma** count photon hits at `HIT_THR` in the spot, against the dark.
- **SNR Cameron (ROI)** is the thesis convention over the whole ROI. It is diluted for a small spot.

The image grid uses one colour scale, normalised per 1000 gates.

In [ ]:
pairs = ht.auto_pair(ht.SESSION_RUNS)
table = ht.summary_table(pairs, spot=SPOT, threshold=HIT_THR)
display(table)
ht.FIG_DIR.mkdir(parents=True, exist_ok=True)
table.to_csv(ht.FIG_DIR / "session_summary.csv", index=False)
with_dark = [(r, d) for r, d in pairs if d is not None]
if with_dark:
    ht.plot_compare(with_dark, spot=SPOT, save_as="session_side_by_side")

## 19. Shut down

This sets the lowest gain, turns the intensifier off, switches to the internal trigger and closes the camera, so LightField can open it. All runs are already on disk.

In [ ]:
ht.shutdown(cam)
del cam